<a href="https://colab.research.google.com/github/sultanjacob/Applied-Machine-Learning/blob/main/14_Deep_Learning/04_Model_Quantization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Phase 15: Production Engineering
#### Part 01: Post-Training Quantization

To deploy our neural networks to general edge servers and production APIs, we must compress them. Post-training quantization converts the standard 32-bit floating-point weights into highly optimized 8-bit integers.

This transformation drastically reduces the memory footprint and accelerates HTTP API inference latency, allowing our models to process live environmental data at scale with virtually no loss in mathematical accuracy. We will convert all three `.keras` models into optimized `.tflite` engines.

In [5]:
import tensorflow as tf
import os

# Ensure models directory exists
os.makedirs('models', exist_ok=True)

def quantize_model(keras_path, tflite_path, is_lstm=False):
    print(f"⏳ Converting {keras_path} to optimized TFLite...")

    # 1. Load the original Keras model
    model = tf.keras.models.load_model(keras_path)

    # 2. Initialize the converter
    converter = tf.lite.TFLiteConverter.from_keras_model(model)

    # 3. Apply dynamic range quantization
    converter.optimizations = [tf.lite.Optimize.DEFAULT]

    # 4. Apply LSTM specific operational support
    if is_lstm:
        converter.target_spec.supported_ops = [
            tf.lite.OpsSet.TFLITE_BUILTINS,
            tf.lite.OpsSet.SELECT_TF_OPS
        ]
        converter._experimental_lower_tensor_list_ops = False

    # 5. Execute conversion
    tflite_model = converter.convert()

    # 6. Save the production-ready model
    with open(tflite_path, 'wb') as f:
        f.write(tflite_model)

    # Calculate and display the compression metrics
    keras_size = os.path.getsize(keras_path) / 1024
    tflite_size = os.path.getsize(tflite_path) / 1024
    print(f"✅ Saved to: {tflite_path}")
    print(f"📉 Size reduction: {keras_size:.1f} KB -> {tflite_size:.1f} KB\n")

# Define the corrected model conversion mapping
models_to_convert = [
    ('models/mlp_carbon_flux_v1.keras', 'models/mlp_carbon_flux_quantized.tflite', False),
    ('models/cnn_spatial_vision_v1.keras', 'models/cnn_spatial_quantized.tflite', False),
    ('models/lstm_temporal_forecast_v1.keras', 'models/lstm_temporal_quantized.tflite', True)
]

# Execute the pipeline
for keras_file, tflite_file, is_lstm in models_to_convert:
    if os.path.exists(keras_file):
        quantize_model(keras_file, tflite_file, is_lstm)
    else:
        print(f"⚠️ Warning: '{keras_file}' not found. Make sure all previous notebooks successfully saved their models.")

⏳ Converting models/mlp_carbon_flux_v1.keras to optimized TFLite...
Saved artifact at '/tmp/tmpaw1mv4l2'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 8), dtype=tf.float32, name='input_layer_1')
Output Type:
  TensorSpec(shape=(None, 1), dtype=tf.float32, name=None)
Captures:
  136534258056272: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136531495849616: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136531495861904: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136531495852496: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136531495852112: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136531495853072: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136531495852880: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136531495853456: TensorSpec(shape=(), dtype=tf.resource, name=None)
✅ Saved to: models/mlp_carbon_flux_quantized.tflite
📉 Size reduction: 45.8 KB -> 6.3 KB

⏳ Conver